# Installing NRPy and Running a First Standalone Project

This notebook verifies an NRPy install, generates the standalone Cartesian wave-equation project, builds it, runs it, and inspects the first diagnostic file. No numerical relativity background is assumed.

## Table of Contents

1. Step 1: Prerequisites and install paths
2. Step 2: Verify the NRPy import path
3. Step 3: Generate the Cartesian wave-equation project
4. Step 4: Inspect the generated tree
5. Step 5: Build the generated project
6. Step 6: Run the executable and inspect diagnostics
7. Step 7: Interpret the first success criterion
8. Step 8: Where this fits in the 2-torial

## Step 1: Prerequisites and Install Paths

Learners install the latest package:

```bash
python -m pip install nrpy
```

Repository-coupled authors in this workspace use:

```bash
cd /work/nrpy
python -m pip install -U -r requirements-dev.txt
python -m pip install -e .
```

These installation commands are intentionally not run automatically. The import-path check below verifies which NRPy Python is using. Generation requires Python; build and run steps require `make` and a C compiler.

## Step 2: Verify the NRPy Import Path

This cell prints the imported NRPy package path. Repository-coupled authors should see the local checkout after the editable install.

In [ ]:
from pathlib import Path

import nrpy

nrpy_path = Path(nrpy.__file__).resolve()
print(nrpy_path)

local_nrpy_package = Path("/work/nrpy/nrpy").resolve()
if local_nrpy_package.exists():
    try:
        nrpy_path.relative_to(local_nrpy_package)
        using_local_checkout = True
    except ValueError:
        using_local_checkout = False
    if not using_local_checkout:
        print(
            "Note for repository authors: this kernel is not importing NRPy from "
            "/work/nrpy. Use the editable install commands in Step 1 for "
            "source-coupled authoring."
        )

## Helper Functions and Workspace Paths

The remaining cells use these paths and helpers to run commands, inspect text files, and prevent accidental deletion of existing generated projects.

In [ ]:
import os
from pathlib import Path
import shutil
import subprocess
import sys

PROJECT_NAME = "wave_equation_cartesian"
WORKSPACE_ROOT = Path.cwd().resolve()
LOCAL_NRPY_ROOT = Path("/work/nrpy").resolve()
GENERATOR_CWD = (
    LOCAL_NRPY_ROOT
    if (LOCAL_NRPY_ROOT / "nrpy" / "examples" / "wave_equation_cartesian.py").exists()
    else WORKSPACE_ROOT
)
PROJECT_DIR = GENERATOR_CWD / "project" / PROJECT_NAME

print(f"Generator working directory: {GENERATOR_CWD}")
print(f"Project directory: {PROJECT_DIR}")


def run_command(command, cwd, max_lines=40):
    cwd = Path(cwd)
    if not cwd.is_dir():
        raise FileNotFoundError(f"Working directory does not exist: {cwd}")

    print(f"$ {' '.join(str(part) for part in command)}")
    print(f"cwd: {cwd}")
    try:
        completed = subprocess.run(
            command,
            cwd=cwd,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            check=True,
        )
    except subprocess.CalledProcessError as err:
        output = err.stdout or ""
        lines = output.splitlines()
        if lines:
            print(f"Last {min(max_lines, len(lines))} output line(s):")
            print("\n".join(lines[-max_lines:]))
        raise

    output = completed.stdout or ""
    lines = output.splitlines()
    if lines:
        print(f"Last {min(max_lines, len(lines))} output line(s):")
        print("\n".join(lines[-max_lines:]))
    else:
        print("(no output)")
    return completed


def require_tool(name):
    resolved = shutil.which(name)
    if resolved is None:
        raise RuntimeError(f"Required tool not found on PATH: {name}")
    print(f"{name}: {resolved}")
    return resolved


def first_lines(path, n=8):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    print(f"First {n} line(s) of {path}:")
    with path.open("r", encoding="utf-8", errors="replace") as stream:
        for _, line in zip(range(n), stream):
            print(line.rstrip())


def ensure_safe_to_regenerate(project_dir):
    project_dir = Path(project_dir)
    if not project_dir.exists() and not project_dir.is_symlink():
        return
    if project_dir.is_symlink() or not project_dir.is_dir():
        raise RuntimeError(
            f"{project_dir} already exists and would be overwritten by regeneration. "
            "Move or delete it before rerunning this notebook."
        )

    sample = []
    for child in project_dir.iterdir():
        suffix = "/" if child.is_dir() else ""
        sample.append(f"{child.name}{suffix}")
        if len(sample) >= 12:
            break
    preview = ", ".join(sample) if sample else "(empty directory)"
    raise RuntimeError(
        f"{project_dir} already exists. The NRPy generator recreates this "
        f"directory before writing a new project. Sample existing entries: {preview}. "
        "Move or delete it before rerunning this notebook."
    )

## Step 3: Generate the Cartesian Wave-Equation Project

The generator writes `project/wave_equation_cartesian/` relative to `GENERATOR_CWD`. It recreates that directory, so this notebook refuses to run the generator when the target already exists.

In [ ]:
ensure_safe_to_regenerate(PROJECT_DIR)
run_command(
    [sys.executable, "-m", "nrpy.examples.wave_equation_cartesian"],
    cwd=GENERATOR_CWD,
)
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(PROJECT_DIR)
print(PROJECT_DIR)

## Step 4: Inspect the Generated Tree

A standalone BHaH project includes C sources, headers, a `Makefile`, a parameter file, helper directories, and later an executable and diagnostics.

In [ ]:
required_paths = [
    PROJECT_DIR / "Makefile",
    PROJECT_DIR / f"{PROJECT_NAME}.par",
]
missing = [path for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing generated file(s): {missing}")

c_sources = sorted(PROJECT_DIR.rglob("*.c"))
headers = sorted(PROJECT_DIR.rglob("*.h"))
if not c_sources:
    raise FileNotFoundError(f"No generated C sources found under {PROJECT_DIR}")
if not headers:
    raise FileNotFoundError(f"No generated headers found under {PROJECT_DIR}")

print("Required files:")
for path in required_paths:
    print(f"  {path.relative_to(PROJECT_DIR)}")
print(f"C sources: {len(c_sources)}")
print(f"Headers: {len(headers)}")

generated_files = sorted(
    path.relative_to(PROJECT_DIR)
    for path in PROJECT_DIR.rglob("*")
    if path.is_file()
)
print("Generated file sample:")
for relpath in generated_files[:25]:
    print(f"  {relpath}")
if len(generated_files) > 25:
    print(f"  ... {len(generated_files) - 25} more file(s)")

first_lines(PROJECT_DIR / f"{PROJECT_NAME}.par", n=8)

## Step 5: Build the Generated Project

This step requires `make` and a C compiler. If either is unavailable, project generation can still be valid, but full first-run validation cannot be completed.

In [ ]:
require_tool("make")

cc_env = os.environ.get("CC")
if cc_env:
    cc_path = shutil.which(cc_env)
    if cc_path is None:
        raise RuntimeError(f"CC is set to {cc_env!r}, but it does not resolve on PATH.")
    print(f"CC: {cc_env} -> {cc_path}")
else:
    for compiler in ("cc", "gcc", "clang"):
        print(f"{compiler}: {shutil.which(compiler) or 'not found'}")

run_command(["make"], cwd=PROJECT_DIR)

executable = PROJECT_DIR / PROJECT_NAME
if not executable.is_file():
    raise FileNotFoundError(executable)
print(f"Executable: {executable}")

## Step 6: Run the Executable and Inspect Diagnostics

Run the default case, then rerun with `2.0` as the `convergence_factor` command-line override. The second run should write `out0d-conv_factor2.00.txt`.

In [ ]:
executable = PROJECT_DIR / PROJECT_NAME
if not executable.is_file():
    raise FileNotFoundError(executable)

run_command([f"./{PROJECT_NAME}"], cwd=PROJECT_DIR)
run_command([f"./{PROJECT_NAME}", "2.0"], cwd=PROJECT_DIR)

diagnostic = PROJECT_DIR / "out0d-conv_factor2.00.txt"
if not diagnostic.is_file():
    raise FileNotFoundError(diagnostic)
first_lines(diagnostic, n=8)

## Step 7: Interpret the First Success Criterion

The first setup milestone is complete when generation succeeds, `make` succeeds, the executable runs, and diagnostic files appear. Deeper convergence analysis belongs in later wave-equation and validation notebooks. The `convergence_factor` value used above is a command-line override registered by the generator.

Rerun the generator rather than hand-editing generated C files when changing the generated project.

## Step 8: Where This Fits in the 2-torial

This setup unit leads into the package map and repository layout, generated-project anatomy, core APIs (`params`, `grid`, `indexedexp`, `finite_difference`, `c_codegen`, `c_function`), wave-equation notebooks, and later reference-metric and BHaH workflows.